<a href="https://colab.research.google.com/github/lili-codelab/comp-linguistics/blob/main/li_fine_tuning_hw_ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#### Домашнее задание

**Датасет:** [`ag_news`](https://huggingface.co/datasets/fancyzhx/ag_news) — классификация новостей по 4-м категориям (World, Sports, Business, Sci/Tech)

**Техническое задание:**

1.  Загрузите датасет `ag_news`
2.  Выберите модель для дообучения (например, `distilbert-base-uncased` или `bert-base-uncased`), `num_labels=4`
3.  Токенизируйте данные (`max_length=128`)
4.  Настройте `TrainingArguments`:
    *   `learning_rate = 2e-5`
    *   `per_device_train_batch_size = 16`
    *   `num_train_epochs = 3`
    *   `eval_strategy = "epoch"`
    *   `save_strategy = "epoch"`
    *   `load_best_model_at_end = True`
    *   `metric_for_best_model = "accuracy"`
5.  Обучите модель с помощью `Trainer`. Для метрик используйте `evaluate.load("accuracy")`
6.  Выведите accuracy на тестовой выборке
7.  Сохраните модель в папку `./ag_news_model`
8.  Протестируйте модель на трех новых новостях (вписать вручную), используя `pipeline`. Выведите предсказанный класс и уверенность модели

In [2]:
!pip install transformers datasets evaluate accelerate gradio -q
!pip install huggingface_hub -q

import torch
print(f"GPU доступен: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Тип GPU: {torch.cuda.get_device_name(0)}")


import numpy as np
import torch
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding
)
from datasets import load_dataset
import evaluate

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.7 MB/s eta 0:00:00
GPU доступен: True
Тип GPU: Tesla T4


In [9]:
# 1
dataset = load_dataset("ag_news")
print(f"Датасет загружен. Train: {len(dataset['train'])}, Test: {len(dataset['test'])}")

Датасет загружен. Train: 120000, Test: 7600


In [10]:
# 2
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=4
).to(device)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [13]:
# 3
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=128)

tokenized_datasets = dataset.map(tokenize_function, batched=True)
train_dataset = tokenized_datasets["train"]
eval_dataset = tokenized_datasets["test"].shuffle(seed=42).select(range(5000))

In [14]:
# 4
training_args = TrainingArguments(
    output_dir="./ag_news_model",
    learning_rate = 2e-5,
    per_device_train_batch_size = 16,
    num_train_epochs = 3,
    eval_strategy = "epoch",
    save_strategy = "epoch",
    load_best_model_at_end = True,
    metric_for_best_model = "accuracy"
)

In [15]:
# 5
accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return accuracy_metric.compute(predictions=predictions, references=labels)


trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = train_dataset,
    eval_dataset = eval_dataset,
    data_collator = DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics = compute_metrics,
)

trainer.train()

Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [16]:
# 6
results = trainer.evaluate()
print(f"\nEvaluation results: {results}")

Epoch,Training Loss,Validation Loss,Accuracy
0,0.229352,0.214369,0.929000



Evaluation results: {'eval_loss': 0.21436911821365356, 'eval_accuracy': 0.929}


In [17]:
# 8
from transformers import pipeline

classifier = pipeline("sentiment-analysis", model = model, tokenizer = tokenizer)

test_texts = [
    "Max Verstappen expressed frustration with his 'undriveable' 2026 F1 car at the Chinese GP, while young talent Antonelli secured a dramatic pole position.",
    "Silver Lake acquired the remaining shares of WWE/UFC parent company Endeavor for billion. Global Sportstech market is projected to reach billion by 2034, driven by wearable technology and fan engagement.",
    "Apple's MacBook Neo is the most repairable laptop the company has released since 2014 according to iFixit analysis with easily replaceable battery and keyboard but soldered RAM limits future upgrades.",
    "Scientists have developed a new perovskite-based gamma-ray detector, promising to advance nuclear medicine imaging technology."
]

for text in test_texts:
    result = classifier(text)
    print(f"Text: {text}\nPredicted class: {result[0]['label']}, Confidence: {result[0]['score']}\n")

Text: Max Verstappen expressed frustration with his 'undriveable' 2026 F1 car at the Chinese GP, while young talent Antonelli secured a dramatic pole position.
Predicted class: LABEL_1, Confidence: 0.9945705533027649

Text: Silver Lake acquired the remaining shares of WWE/UFC parent company Endeavor for billion. Global Sportstech market is projected to reach billion by 2034, driven by wearable technology and fan engagement.
Predicted class: LABEL_2, Confidence: 0.8708838820457458

Text: Apple's MacBook Neo is the most repairable laptop the company has released since 2014 according to iFixit analysis with easily replaceable battery and keyboard but soldered RAM limits future upgrades.
Predicted class: LABEL_3, Confidence: 0.966253936290741

Text: Scientists have developed a new perovskite-based gamma-ray detector, promising to advance nuclear medicine imaging technology.
Predicted class: LABEL_3, Confidence: 0.9655899405479431



Комментарий к работе: я начала работу довольно поздно.. в первый раз обучение в пятом пункте прервалось по тех причинам, вероятно, из-за нестабильного подключения. после этого я уже не успевала дождаться полного обучения.

Спасибо за работу!